# Uniform B-Splines

Parametric piece-wise curve, determined by set of control points and associated basis polinomial functions of degree $d$:

$$S^{(d)}(t): \mathbb{R} \to \mathbb{R}^3 \tag{curve}$$

Parametric basis functions are the polynomials of degree $d$:

$$B^{(d)}(t): \mathbb{R} \to \mathbb{R} \tag{basis}$$

In general case:

- $t$ runs along path infinitely
- The path is subdivided into knots $t_j$
- A $j$'th segment of the curve corresponds to a span of path with $t \in [t_{i}, t_{i+1})$
- A $j$'th segment is affected by $d+1$ nearby control points and assosiated basis functions
- Each basis function is associated with a control point and determines its weight

The _nearby_ is arbitrary mapping between knots and controls.

In uniform (unit-spaced) case:

- $0 \lt t_{local} \lt 1$ for each segment
- Evaluation of (sum of) all B-functions are all the same for each segment.
- Each segment is determined by a set of control points and (sum of) basis functions


# Generic matrix form

$$
B^{(d)}(t) = [1, t^1, \dots, t^d] \; M^{(d)}
$$

$$
S_j^{(d)}(t) = [1, t^1, \dots, t^d] \; M^{(d)} \; C^{(d)}_j
$$

The $[1, t^1, \dots, t^d]$ is a power basis simply to isolate dependency on t

The $B^{(d)}$ is a row vector of $d+1$ basis functions affecting curve at a point. They are the same for every segment.

The $C^{(d)}$ is not a real matrix, it's a column-vector of $d+1$ control points in $\mathbb{R}^2$ or $\mathbb{R}^3$ (or anything else in general), affecting curve at a particular segment $j$


## The B-matrices

$$
M^{(1)} =
\begin{pmatrix}
1 & 0 \\
-1 & 1
\end{pmatrix}
\tag{linear}
$$

The linear case is the same as LERP: $[1, t] M C = [1, t] \big[ c_1, c_2 - c_1\big]^T = c_1 + (c_2 - c_1) t = c_1 (1 - t) + c_2 (t)$

$$
M^{(2)} = \frac{1}{2}
\begin{pmatrix}
1 & 1 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1
\end{pmatrix}
\tag{quadratic}
$$

$$
M^{(3)} = \frac{1}{6}
\begin{pmatrix}
1 & 4 & 1 & 0 \\
-3 & 0 & 3 & 0 \\
3 & -6 & 3 & 0 \\
-1 & 3 & -3 & 1
\end{pmatrix}
\tag{cubic}
$$


In [ ]:
import numpy as np
from numpy.typing import NDArray

Vec = np.ndarray[tuple[int], np.dtype[np.floating]]  # 2D or 3D
vec = lambda *args: np.array(args)
nparr = lambda *args: np.array(args)
npgen = lambda iter: np.array(tuple(iter))

In [ ]:
def powers(d: int, t: float) -> NDArray:
    """Calc power basis 1×(d+1) incuding 0"""
    return np.power(t, np.arange(d + 1))


def powersnp(d: int, t: NDArray) -> NDArray:
    """Calc power basis num×(d+1) incuding 0"""
    return np.power(t[:, None], np.arange(d + 1))

In [ ]:
M_ = {
    1: np.array([[1, 0], [-1, 1]]),
    2: np.array([[1, 1, 0], [-2, 2, 0], [1, -2, 1]]) / 2,
    3: np.array([[1, 4, 1, 0], [-3, 0, 3, 0], [3, -6, 3, 0], [-1, 3, -3, 1]]) / 6,
}


# keeping them same d
Mdt_ = {
    1: np.array([[-1, 1], [0, 0]]),
    2: np.array([[-1, 1, 0], [1, -2, 1], [0, 0, 0]]),
    3: np.array([[-1, 0, 1, 0], [2, -4, 2, 0], [-1, 3, -3, 1], [0, 0, 0, 0]]) / 2,
}

In [ ]:
def iter_controls(d: int, controls: NDArray):
    """Iterate over all possible segments and yield their control points
    (d+1 points for each segment)
    """
    num = controls.shape[0]
    for i in range(num - d):
        yield controls[i : i + d + 1]

In [ ]:
def random_points(num: int, jigg=0.25):
    """Geneates control points
    first and ast fized at corners (1, 0) and (0, 1),
    midpoints jiggled
    """
    xx = np.linspace(1.0, 0.0, num)
    yy = np.linspace(0.0, 1.0, num)
    zz = np.zeros(num)

    coords = np.stack((xx, yy, zz)).T
    coords[1:-1] += np.random.randn(num - 2, 3) * jigg
    return coords

#### The curve

All points of shape (num, dim)


In [ ]:
def spline_point(d: int, controls: NDArray, t: float):
    """A point on a curve segment on the controls
    - t in [0, 1]
    """
    assert controls.shape[0] == d + 1
    return powers(d, t) @ M_[d] @ controls


def spline_curve(d: int, controls: NDArray, tspace: NDArray):
    """All points on a curve segment on the controls
    - for all t in local tspace 0 <= t <= 1
    - the controls should be of d
    """
    assert controls.shape[0] == d + 1
    B = powersnp(d, tspace) @ M_[d]
    return B @ controls


def spline_flow(d: int, controls: NDArray, tspace: NDArray):
    """Tangential velocity of a curve segment at t
    - t in [0, 1]
    - the controls should be of d
    """
    assert controls.shape[0] == d + 1
    Bdt = powersnp(d, tspace) @ Mdt_[d]
    return Bdt @ controls

In [ ]:
def multispline_curve(d: int, controls: NDArray, tspace: NDArray):
    """All points of curve concatenated"""
    assert controls.shape[1] == d + 1
    B = powersnp(d, tspace) @ M_[d]
    return np.concat([B @ ctrl for ctrl in iter_controls(d, controls)])


def multispline_flow(d: int, controls: NDArray, tspace: NDArray):
    assert len(controls) > d + 1
    Bdt = powersnp(d, tspace) @ Mdt_[d]
    return np.concat([Bdt @ ctrl for ctrl in iter_controls(d, controls)])

---


In [ ]:
from ipywidgets import widgets as wg
import plotly.graph_objects as go

### Plotting Basis


In [ ]:
# D = 3
# tspace = np.linspace(0, 1, 16)
# bb = powersnp(D, tspace) @ M_[D]

In [ ]:
# go.Figure(
#     data=[go.Scatter(x=tspace, y=b, mode="lines") for b in bb.T],
#     layout=dict(title=f"Basis functions d={D}"),
# )

### Plotting curves


In [ ]:
# D = 3
# controls = random_points(D + 1)
# tspace = np.linspace(0.0, 1.0, 10)
# curve = spline_curve(D, controls, tspace)

In [ ]:
# go.Figure(
#     data=[
#         go.Scatter(x=controls[:,0], y=controls[:,1], mode="markers+lines", name="points"),
#         go.Scatter(x=curve[:,0], y=curve[:,1], mode="lines", name="curve"),
#     ],
#     layout=dict(title=f"Spline segment d={D}"),
# )

In [ ]:
from widgets import Curve3DView, Curve2DView

In [ ]:
view3d = Curve3DView(
    plotting_curve=lambda params, tspace: spline_curve(params[0], params[1], tspace),
    plotting_flow=lambda params, tspace: spline_flow(params[0], params[1], tspace),
)
view2d = Curve2DView(
    plotting_curve=lambda params, tspace: spline_curve(params[0], params[1], tspace),
    plotting_flow=lambda params, tspace: spline_flow(params[0], params[1], tspace),
)

wg.HBox([view3d, view2d])

In [ ]:
D = 3
controls = random_points(D + 1)
view3d.controls = controls
view3d.params = (D, controls)
view2d.controls = controls[:, :2]
view2d.params = (D, controls[:, :2])

# Quadratic Bezier

A segment of B(2)-spline on $p_1, p_2, p_3$ is a quadratic bezier on $c_1 = \frac{p_1 + p_2}{2}, c_2 = p_2, c_3 = \frac{p_3 + p_2}{2}$
With $c_2 = c_j$ at it's zenith.

$$
S(t) = [1, t, t^2]
\begin{pmatrix}
1 & 0 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1 \\
\end{pmatrix}
\begin{bmatrix}
c_1 \\
c_2 \\
c_3 \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t) = [1, t, t^2]
\begin{pmatrix}
-2 & 2 & 0 \\
2 & -4 & 2 \\
0 & 0 & 0 \\
\end{pmatrix}
\begin{bmatrix}
c_1 \\
c_2 \\
c_3 \\
\end{bmatrix}
$$


In [ ]:
def beziefy_controls(p1: NDArray, p2: NDArray, p3: NDArray) -> tuple[NDArray, NDArray, NDArray]:
    """Bezier control points from bspline segment points"""
    return (p1 + p2) * 0.5, p2, (p2 + p3) * 0.5

In [ ]:
B2 = np.array([[1, 0, 0], [-2, 2, 0], [1, -2, 1]])
B2DT = np.array([[-2, 2, 0], [2, -4, 2], [0, 0, 0]])


def b2zier_point(controls: tuple[Vec, Vec, Vec], t: float):
    """A point on a bezier curve between the controls
    - t in [0, 1]
    - the controls should be of 3
    """
    assert len(controls) == 3
    return powers(2, t) @ B2 @ np.array(controls)


def b2zier_curve(controls: tuple[Vec, Vec, Vec], tspace: NDArray):
    """All points on a bezier curve between the controls
    - for all t in local tspace 0 <= t <= 1
    - the controls should be of 3
    """
    assert len(controls) == 3
    return powersnp(2, tspace) @ B2 @ np.array(controls)


def b2zier_flow(controls: tuple[Vec, Vec, Vec], tspace: NDArray):
    """Tangential direction/speed"""
    assert len(controls) == 3
    return powersnp(2, tspace) @ B2DT @ np.array(controls)

In [ ]:
view3d = Curve3DView(
    plotting_curve=lambda params, tspace: b2zier_curve(params, tspace),
    plotting_flow=lambda params, tspace: b2zier_flow(params, tspace),
)
view2d = Curve2DView(
    plotting_curve=lambda params, tspace: b2zier_curve(params, tspace),
    plotting_flow=lambda params, tspace: b2zier_flow(params, tspace),
)

wg.HBox([view3d, view2d])

In [ ]:
points = random_points(3)
view3d.controls = points
view3d.params = beziefy_controls(*points)
view2d.controls = points[:, :2]
view2d.params = beziefy_controls(*points[:, :2])

# Quadratic Beziel

Centering around apex point and shifting $t$.

- $c_2 \to 0$
- $t \in [0, 1] \to t' \in [-0.5, +0.5]$
- $c_1 \to v_1 = c_1 - c_2$
- $c_3 \to v_3 = c_3 - c_2$
- $v_a = v_3 + v_1$ — parabolic axial vector
- $v_t = v_3 - v_1$ — parabolic tangential vector
- $c_a = c_2 + .25 v_a = S(0)$ — apex

Within basis axes $v_t, v_a$ the curve is just a parabola $y = x^2$, skewed.

$$
S(t') - c_a=
\big[1, t', t'^2 \big]
\begin{bmatrix}
0 \\
v_t \\
v_a \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t') =
\big[1, t', t'^2 \big]
\begin{bmatrix}
v_t \\
2 v_a \\
0 \\
\end{bmatrix}
$$

Tho, the transform from/into the $v_t, v_a$ does not preserve perpendicularity and distances.


In [27]:
def beziel_parab(c1: Vec | NDArray, c2: Vec | NDArray, c3: Vec | NDArray):
    """Calculate parabolic basis ca vt va"""
    v1 = c1 - c2
    v3 = c3 - c2
    va = v3 + v1
    vt = v3 - v1
    ca = c2 + 0.25 * va
    return ca, vt, va

In [28]:
def beziel_M(ca: Vec, vt: Vec, va: Vec):
    return np.array([np.zeros(ca.shape), vt, va])


def beziel_Mdt(ca: Vec, vt: Vec, va: Vec):
    return np.array([vt, 2 * va, np.zeros(ca.shape)])

In [29]:
def beziel_curve(params: tuple[Vec, Vec, Vec], tspace: NDArray):
    ca, vt, va = params
    MC = beziel_M(ca, vt, va)
    return powersnp(2, tspace - 0.5) @ MC + ca


def beziel_flow(params: tuple[Vec, Vec, Vec], tspace: NDArray):
    ca, vt, va = params
    MC = beziel_Mdt(ca, vt, va)
    return powersnp(2, tspace - 0.5) @ MC

In [ ]:
view3d = Curve3DView(
    plotting_curve=lambda params, tspace: beziel_curve(params, tspace),
    plotting_flow=lambda params, tspace: beziel_flow(params, tspace),
)
view2d = Curve2DView(
    plotting_curve=lambda params, tspace: beziel_curve(params, tspace),
    plotting_flow=lambda params, tspace: beziel_flow(params, tspace),
)

wg.HBox([view3d, view2d])

In [33]:
controls = random_points(3)
p1, p2, p3 = controls
c1, c2, c3 = beziefy_controls(p1, p2, p3)
view3d.controls = controls
view3d.params = beziel_parab(c1, c2, c3)
view2d.controls = controls[:, :2]
view2d.params = beziel_parab(c1[:2], c2[:2], c3[:2])

## Multicurvie

Continuous spline split into bezier segments


In [34]:
def split_controls(controls: NDArray) -> tuple[NDArray, NDArray, NDArray]:
    """Control points for each segment"""
    num = controls.shape[0]
    return (
        controls[0 : num - 2],
        controls[1 : num - 1],
        controls[2 : num - 0],
    )


def slicetime(tspace: NDArray, j: int):
    """Slice of tspace for a jth segment"""
    return tspace[np.floor(tspace) == j] - j

In [35]:
def bezielm_M(ca: NDArray, vt: NDArray, va: NDArray) -> NDArray:
    """A matrix (dim, 3) or array of matrices (num, dim, 3)"""
    buff = np.zeros((ca.shape[0], 3, ca.shape[1]))
    buff[:, 1] = vt
    buff[:, 2] = va
    return buff


def bezielm_Mdt(ca: NDArray, vt: NDArray, va: NDArray) -> NDArray:
    buff = np.zeros((ca.shape[0], 3, ca.shape[1]))
    buff[:, 0] = vt
    buff[:, 1] = 2 * va
    return buff

In [ ]:
# def bezielm_curve(tspace: NDArray, ca: NDArray, vt: NDArray, va: NDArray) -> NDArray:
#     """Render multiple bezier on the same t space"""
#     W = powersnp(2, tspace - 0.5)  # (len, 3)
#     MCs = bezielm_M(ca, vt, va)  # (num, 3, dim)
#     out = W @ MCs + ca[:, None, :]
#     return out.reshape(-1, out.shape[-1])

# def bezielm_flow(tspace: NDArray, ca: NDArray, vt: NDArray, va: NDArray):
#     W = powersnp(2, tspace - 0.5)  # (len, 3)
#     MCs = bezielm_Mdt(ca, vt, va)  # (num, 3, dim)
#     # return np.concat([W @ M + ca for M, ca in zip(MCs, ca)]) # (num*len, dim)
#     out = W @ MCs
#     return out.reshape(-1, out.shape[-1])

In [36]:
def bezielm_curve(params: tuple[NDArray, NDArray, NDArray], tspace: NDArray):
    ca, vt, va = params
    num = ca.shape[0]
    MCs = bezielm_M(ca, vt, va)
    return np.concat([powersnp(2, slicetime(tspace, j) - 0.5) @ MCs[j] + ca[j] for j in range(num)])


def bezielm_flow(params: tuple[NDArray, NDArray, NDArray], tspace: NDArray):
    ca, vt, va = params
    num = ca.shape[0]
    MCs = bezielm_Mdt(ca, vt, va)
    return np.concat([powersnp(2, slicetime(tspace, j) - 0.5) @ MCs[j] for j in range(num)])


In [39]:
view3d = Curve3DView(
    plotting_curve=lambda params, tspace: bezielm_curve(params, tspace),
    plotting_flow=lambda params, tspace: bezielm_flow(params, tspace),
)
display(view3d)

Curve3DView(children=(VBox(children=(FigureWidget({
    'data': [{'hovertemplate': 'xyz: %{x:.03f} %{y:.03f} %…

In [57]:
N = 7
S = N - 2
controls = random_points(N)
p1, p2, p3 = split_controls(controls)
c1, c2, c3 = beziefy_controls(p1, p2, p3)
ca, vt, va = beziel_parab(c1, c2, c3)

view3d.controls = controls
view3d.params = beziel_parab(c1, c2, c3)
view3d.tspace_curve = np.linspace(0.0, S, S * 64, endpoint=False)
view3d.tspace_flow = np.linspace(0.0, S, S * 3, endpoint=False)